# More on Polars Series
It's often necessary to compute something that involves data from more than one Polars Series. If two Polars Series are of the same size, Python makes it easy to do calculations involving both Series.

For our first example, we return once more to the temperature data.  This time, we create Polars Series of average daily [high](http://berkeleyearth.lbl.gov/auto/Regional/TMAX/Text/global-land-TMAX-Trend.txt) and [low](http://berkeleyearth.lbl.gov/auto/Regional/TMIN/Text/global-land-TMIN-Trend.txt) temperatures for the decades surrounding 1850, 1900, 1950, and 2000.

In [1]:
import polars as pl
path_data = '../../../assets/data/'

In [2]:
baseline_high = 14.48
highs = pl.Series("highs", [baseline_high - 0.880, 
                   baseline_high - 0.093,
                   baseline_high + 0.105, 
                   baseline_high + 0.684])
highs

highs
f64
13.6
14.387
14.585
15.164


In [3]:
baseline_low = 3.00
lows = pl.Series("lows", [baseline_low - 0.872, baseline_low - 0.629,
                  baseline_low - 0.126, baseline_low + 0.728])
lows

lows
f64
2.128
2.371
2.874
3.728


Suppose we'd like to compute the average daily *range* of temperatures for each decade.  That is, we want to subtract the average daily high in the 1850s from the average daily low in the 1850s, and the same for each other decade.

We could write this laboriously using `.item`:

In [4]:
pl.Series("range", [
    highs.item(0) - lows.item(0),
    highs.item(1) - lows.item(1),
    highs.item(2) - lows.item(2),
    highs.item(3) - lows.item(3)
])

range
f64
11.472
12.016
11.711
11.436


As when we converted a Polars Series of temperatures from Celsius to Fahrenheit, Python provides a much cleaner way to write this:

In [5]:
highs - lows

highs
f64
11.472
12.016
11.711
11.436


![Illustration of Polars Series subtraction. The highs Series is shown on the left followed by a subtraction sign and then the lows Series. The Series are aligned so the numbers in the first position of the Series are next to each other, same for the second, third, and fourth positions. Following that is an equal sign and a Series representation with the first entry of lows subtracted from the first entry of highs, and the same for the second, third, and fourth entries. Following that is another equal sign with a Series showing the result of the subtraction in each entry of the Series.](../../../images/array_subtraction.png)

What we've seen in these examples are special cases of a general feature of Polars Series.

## Elementwise arithmetic on pairs of numerical Polars Series
If an arithmetic operator acts on two Polars Series of the same size, then the operation is performed on each corresponding pair of elements in the two Series. The final result is a Polars Series. 

For example, if `series1` and `series2` have the same number of elements, then the value of `series1 * series2` is a Polars Series. Its first element is the first element of `series1` times the first element of `series2`, its second element is the second element of `series1` times the second element of `series2`, and so on.

## Example: Wallis' Formula for $\pi$ ###
The number $\pi$ is important in many different areas of math. Centuries before computers were invented, mathematicians worked on finding simple ways to approximate the numerical value of $\pi$. We have already seen Leibniz's formula for $\pi$. About half a century before Leibniz, the English mathematician [John Wallis](https://en.wikipedia.org/wiki/John_Wallis) (1616-1703) also expressed $\pi$ in terms of simple fractions, as an infinite product.

$$
\pi = 2 \cdot \left( \frac{2}{1}\cdot\frac{2}{3}\cdot\frac{4}{3}\cdot\frac{4}{5}\cdot\frac{6}{5}\cdot\frac{6}{7}\dots \right)
$$

This is a product of "even/odd" fractions. Let's use Polars Series to multiply a million of them, and see if the product is close to $\pi$.

Remember that multiplication can done in any order [[1]](#footnotes), so we can readjust our calculation to:

$$\pi \approx 2 \cdot \left( \frac{2}{1} \cdot \frac{4}{3} \cdot \frac{6}{5} \cdots \frac{1,000,000}{999999} \right) \cdot \left( \frac{2}{3} \cdot \frac{4}{5} \cdot \frac{6}{7} \cdots \frac{1,000,000}{1,000,001} \right)$$

We're now ready to do the calculation. We start by creating a Polars Series of even numbers 2, 4, 6, and so on upto 1,000,000. Then we create two Polars Series of odd numbers: 1, 3, 5, 7, ... upto 999,999, and 3, 5, 7, ... upto 1,000,001.

In [6]:
# POLARS_CHANGE: an eager range is named "literal" unless alias gives it a name
even = pl.int_range(2, 1000001, 2, eager=True).alias("even")
one_below_even = even - 1
one_above_even = even + 1

Remember that `product` multiplies all the elements of a Polars Series together. Now we can calculate Wallis' product, to a good approximation.

The expression below has three pieces:

- `even / one_below_even` divides element by element: 2/1, 4/3, 6/5, and so on.
- `even / one_above_even` divides element by element: 2/3, 4/5, 6/7, and so on.
- Each `.product()` multiplies every value in that Series into one number. Using just the first three values, `(2/1) * (4/3) * (6/5)` is 3.2, and `(2/3) * (4/5) * (6/7)` is about 0.457.

Those two products are then multiplied together and by 2. With only those three values from each Series, `2 * 3.2 * 0.457` is about 2.926. The cell below uses every value, out to 1,000,000.

In [7]:
2 * (even / one_below_even).product() * (even / one_above_even).product()

3.1415910827951143

That's $\pi$ correct to five decimal places.  Wallis clearly came up with a great formula.

## Footnotes
[1] As we saw in the example about Leibniz's formula, when we add  *infinitely* many fractions, the order can matter. The same is true with multiplying fractions, as we are doing here. But our approximation to $\pi$ uses only a large finite number of fractions, so it's okay to multiply the terms in any convenient order.